In [8]:
import torch
import torch.nn as nn 
class PatchEmbbeding(nn.Module):
    def __init__(self, imge_size=224,in_channel=3,patch_size=16):   
        super().__init__()
        self.patch_size=patch_size                                                                      
        self.embedding_size=patch_size*patch_size*in_channel                                            #(16*16*3=768)                                                    
        self.project=nn.Conv2d(in_channel,self.embedding_size,kernel_size=patch_size,stride=patch_size) #Bx3x224x224 --> Bx768x14x14
    def forward(self,x):
        B,C,H,W=x.shape
        x=self.project(x).flatten(2).transpose(1,2)             #Bx768x14x14 --> Bx768x196 --> Bx196x768
        return x
    
class PositionalEncoding(nn.Module):
    def __init__(self,embedding_size,seq_len):
        super().__init__()
        self.pos_vector=nn.Parameter(torch.randn(1,seq_len+1,embedding_size))
    def forward(self,x):
        return x+self.pos_vector

class Attention(nn.Module):
    def __init__(self, embedding_size,num_head):
        super().__init__()
        self.attn=nn.MultiheadAttention(embed_dim=embedding_size,num_heads=num_head)
    def forward(self,x):
        return self.attn(x,x,x)[0]
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim):
        super().__init__()
        self.attn = Attention(embed_dim, num_heads)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.ReLU(),
            nn.Linear(mlp_dim, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x
class ViT(nn.Module):
    def __init__(self,img_size=224,patch_size=16,in_channel=3,embed_dim=768,num_head=8,depth=6,mlp_dim=1024):
        super().__init__()
        self.patch_embbeding=PatchEmbbeding(imge_size=img_size,in_channel=in_channel,patch_size=16)
        self.positional_vector=PositionalEncoding(embedding_size=embed_dim,seq_len=(img_size//patch_size)**2)
        self.TransformerBlock=nn.ModuleList([TransformerEncoderBlock(embed_dim=embed_dim,num_heads=num_head,mlp_dim=mlp_dim) for i in range(depth)])
        self.cls_token=nn.Parameter(torch.randn(1,1,embed_dim))
        self.mlp_head=nn.Linear(embed_dim,mlp_dim)
    def forward(self,x):
        B=x.size(0)
        x=self.patch_embbeding(x)
        cls_tokens=self.cls_token.expand(B,-1,-1)
        x=torch.cat((cls_tokens,x),dim=1)
        x=self.positional_vector(x)
        for block in self.TransformerBlock:
            x=block(x)
        x=self.mlp_head(x)
        return x
    

In [ ]:
model=ViT(img_size=224,patch_size=16,in_channel=3,embed_dim=768,num_head=8,depth=6,mlp_dim=1024)
x=torch.randn(1,3,224,224)
y=model(x)
print(y.shape)

torch.Size([1, 197, 1024])


: 